# 06 — Model Experiments (Stage 3)

## Goal
Train 7 different models on the engineered features from `05.feature_engineering.ipynb`,
compare them on the same train/test split, and pick the best one for the R² ≥ 0.80
deploy bar.

## The 7 models

| # | Model | Why we're trying it |
|---|---|---|
| 1 | LinearRegression | Control — confirm Stage 2's 0.483 reproduces on the same data |
| 2 | Ridge | L2 regularization — shrinks coefficients, handles multicollinearity in 30 brand dummies |
| 3 | Lasso | L1 regularization — automatic feature selection; may drop noise features |
| 4 | ElasticNet | L1+L2 mix — when Ridge doesn't shrink enough or Lasso over-shrinks |
| 5 | DecisionTreeRegressor | Non-linear baseline — no linearity assumption, no scaling needed |
| 6 | RandomForestRegressor | Bagging ensemble — usually beats single tree significantly |
| 7 | XGBoostRegressor | Boosting — primary target for tabular regression, expected best |

## Targets
- **Deploy bar:** R² ≥ 0.80 on test set
- **Stage 1.5 (raw features, linear):** R² = 0.471
- **Stage 2 (engineered features, linear):** R² = 0.483
- **Gap to close:** 0.32 R² points

## Decision: train ALL models on `log_price`, evaluate in rupee space
- Consistency across the 7 models — easier to compare
- Tree models don't *need* log transform but don't mind it
- Same `np.exp()` to convert predictions back to rupees

## What this notebook does NOT do
- SHAP explainability (Stage 4)
- Build the inference function (Stage 6)
- Build the Streamlit app (Stage 7)


In [ ]:
# install xgboost 
# %pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 3: imports
import warnings
warnings.filterwarnings('ignore')

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import joblib

print("All imports loaded.")


All imports loaded.


### ***Load the engineered parquet file***

In [3]:
# Cell 4: load engineered features + record baselines
data_path = Path(r"d:/Study/data_science/underpriced-listing-predictor/data/04.features/ac_features.parquet")
data = pd.read_parquet(data_path)

print(f"Shape: {data.shape}")
print(f"\nDtype counts:")
print(data.dtypes.value_counts())
print(f"\nTotal null cells: {data.isna().sum().sum()}")

print(f"\nColumns ({len(data.columns)}):")
for col in data.columns:
    print(f"  - {col}")

print("\nFirst 3 rows:")
data.head(3)

# Baselines from earlier stages — these are what we need to beat
print("\n" + "="*60)
print("BASELINES (from earlier stages, evaluated in rupee space):")
print("="*60)
print("  Stage 1.5 — raw features, linear model:    R² = 0.471, MAE = ₹5,255")
print("  Stage 2   — engineered features, linear:   R² = 0.483, MAE = ₹4,754")
print("  Target    — deploy bar:                    R² ≥ 0.80")
print("  Gap to close: 0.32 R² points")


Shape: (994, 28)

Dtype counts:
int64      17
str         3
float64     3
float32     2
Int64       2
int32       1
Name: count, dtype: int64

Total null cells: 0

Columns (28):
  - brand
  - model_year
  - product
  - price
  - user_rating
  - ac_type
  - capacity
  - Dehumidification
  - Turbo Mode
  - Air Swing
  - Self Diagnosis
  - Memory Feature
  - LED Panel Display
  - Night Glow Buttons
  - Wi-Fi Connectivity
  - APP Control
  - Auto Clean
  - Hidden Panel Display
  - Voice Control
  - PM 2.5 Filter
  - inverter
  - star_rating
  - log_price
  - smart_features
  - model_year_missing
  - age
  - inverter_x_star
  - feature_count

First 3 rows:

BASELINES (from earlier stages, evaluated in rupee space):
  Stage 1.5 — raw features, linear model:    R² = 0.471, MAE = ₹5,255
  Stage 2   — engineered features, linear:   R² = 0.483, MAE = ₹4,754
  Target    — deploy bar:                    R² ≥ 0.80
  Gap to close: 0.32 R² points


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 994 entries, 0 to 993
Data columns (total 28 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand                 994 non-null    str    
 1   model_year            994 non-null    float64
 2   product               994 non-null    str    
 3   price                 994 non-null    int64  
 4   user_rating           994 non-null    float32
 5   ac_type               994 non-null    str    
 6   capacity              994 non-null    float32
 7   Dehumidification      994 non-null    int64  
 8   Turbo Mode            994 non-null    int64  
 9   Air Swing             994 non-null    int64  
 10  Self Diagnosis        994 non-null    int64  
 11  Memory Feature        994 non-null    int64  
 12  LED Panel Display     994 non-null    int64  
 13  Night Glow Buttons    994 non-null    int64  
 14  Wi-Fi Connectivity    994 non-null    int64  
 15  APP Control           994 non-null

### ***feature/target setup + train-test split + preprocessor***

In [7]:
# Cell 5: feature/target setup + train-test split + preprocessor

# Target: log_price (consistent with Stage 2)
target = 'log_price'

# Drop columns that aren't features
# - price, log_price: target variables
# - model_year: replaced by age + model_year_missing in Stage 2
# - product: constant "AC" in all 994 rows (useless)
# - Wi-Fi Connectivity, APP Control, Voice Control: combined into smart_features
drop_cols = ['price', 'log_price', 'model_year', 'product',
             'Wi-Fi Connectivity', 'APP Control', 'Voice Control']

# Identify feature columns
feature_cols = [c for c in data.columns if c not in drop_cols + [target]]
print(f"Target: {target}")
print(f"Dropped: {drop_cols}")
print(f"\nFeature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col} ({data[col].dtype})")

# Categorical vs numeric
categorical_cols = data[feature_cols].select_dtypes(include='object').columns.tolist()
numeric_cols = data[feature_cols].select_dtypes(exclude='object').columns.tolist()

print(f"\nCategorical ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")

# Train-test split (same random_state=42 as Stage 2)
X = data[feature_cols]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape},  y={y_test.shape}")
print(f"Train target mean: {y_train.mean():.3f}, std: {y_train.std():.3f}")
print(f"Test target mean:  {y_test.mean():.3f},  std: {y_test.std():.3f}")

# Preprocessor: one-hot brand, one-hot ac_type, passthrough numeric
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_brand', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['brand']),
        ('cat_ac_type', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['ac_type']),
        ('num', 'passthrough', numeric_cols),
    ],
    remainder='drop'
)

print(f"\nPreprocessor defined.")
print(f"  brand → one-hot")
print(f"  ac_type → one-hot")
print(f"  {len(numeric_cols)} numeric columns → passthrough")


Target: log_price
Dropped: ['price', 'log_price', 'model_year', 'product', 'Wi-Fi Connectivity', 'APP Control', 'Voice Control']

Feature columns (21):
  - brand (str)
  - user_rating (float32)
  - ac_type (str)
  - capacity (float32)
  - Dehumidification (int64)
  - Turbo Mode (int64)
  - Air Swing (int64)
  - Self Diagnosis (int64)
  - Memory Feature (int64)
  - LED Panel Display (int64)
  - Night Glow Buttons (int64)
  - Auto Clean (int64)
  - Hidden Panel Display (int64)
  - PM 2.5 Filter (int64)
  - inverter (int32)
  - star_rating (Int64)
  - smart_features (int64)
  - model_year_missing (int64)
  - age (float64)
  - inverter_x_star (Int64)
  - feature_count (int64)

Categorical (2): ['brand', 'ac_type']
Numeric (19): ['user_rating', 'capacity', 'Dehumidification', 'Turbo Mode', 'Air Swing', 'Self Diagnosis', 'Memory Feature', 'LED Panel Display', 'Night Glow Buttons', 'Auto Clean', 'Hidden Panel Display', 'PM 2.5 Filter', 'inverter', 'star_rating', 'smart_features', 'model_year_

### ***Regression Model Comparisons***

In [8]:
# Cell 6: train_and_evaluate() helper
# Wraps a model in the preprocessor pipeline, runs GridSearchCV,
# evaluates in rupee space, returns a results dict for comparison.

def train_and_evaluate(
    name,
    model,
    param_grid,
    cv=3,
    verbose=True
):
    """
    Train a model with GridSearchCV, evaluate in rupee space, return metrics.

    Returns a dict with: name, best_params, fit_time, cv_score_log,
    train/test metrics, predictions, and the fitted best model.
    """
    # Wrap in pipeline
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Grid search (CV in log space, default R²)
    t0 = time.time()
    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv,
        scoring='r2',
        n_jobs=-1,
        refit=True
    )
    grid.fit(X_train, y_train)
    fit_time = time.time() - t0

    # Best model
    best_model = grid.best_estimator_
    best_params = grid.best_params_

    # Predict in log space, convert to rupee space
    y_train_pred = np.exp(best_model.predict(X_train))
    y_test_pred = np.exp(best_model.predict(X_test))
    y_train_true = np.exp(y_train)
    y_test_true = np.exp(y_test)

    # Metrics in rupee space
    train_r2 = r2_score(y_train_true, y_train_pred)
    test_r2 = r2_score(y_test_true, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    test_mae = mean_absolute_error(y_test_true, y_test_pred)
    gap = train_r2 - test_r2

    if verbose:
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        print(f"Best params: {best_params}")
        print(f"Fit time: {fit_time:.2f}s")
        print(f"CV best R² (log space): {grid.best_score_:.4f}")
        print(f"\n  Train (₹): R²={train_r2:.4f}, RMSE=₹{train_rmse:,.0f}, MAE=₹{train_mae:,.0f}")
        print(f"  Test  (₹): R²={test_r2:.4f}, RMSE=₹{test_rmse:,.0f}, MAE=₹{test_mae:,.0f}")
        print(f"  Train-test gap: {gap:+.4f}")

    return {
        'name': name,
        'best_params': best_params,
        'fit_time': fit_time,
        'cv_score_log': grid.best_score_,
        'train_r2': train_r2, 'test_r2': test_r2,
        'train_rmse': train_rmse, 'test_rmse': test_rmse,
        'train_mae': train_mae, 'test_mae': test_mae,
        'gap': gap,
        'y_train_pred': y_train_pred,
        'y_test_pred': y_test_pred,
        'y_train_true': y_train_true,
        'y_test_true': y_test_true,
        'model': best_model,
    }

# Container to collect all model results
results = []

print("Helper function defined. Ready to train models.")


Helper function defined. Ready to train models.


### **Linear Models**

### ***1. Linear Regression***

In [9]:
# Cell 7: LinearRegression (control)
# Note: this notebook uses 19 numeric features (vs 17 in Stage 2), so the
# result may be slightly higher than 0.483. The +2 features are user_rating
# and one of the binary feature columns we kept here but Stage 2 dropped.

model_lr = LinearRegression()

param_grid_lr = {
    'model__fit_intercept': [True, False],
}

result_lr = train_and_evaluate(
    name='LinearRegression (control)',
    model=model_lr,
    param_grid=param_grid_lr,
    cv=3
)

results.append(result_lr)



  LinearRegression (control)
Best params: {'model__fit_intercept': True}
Fit time: 3.52s
CV best R² (log space): 0.4929

  Train (₹): R²=0.5401, RMSE=₹7,188, MAE=₹4,540
  Test  (₹): R²=0.4831, RMSE=₹7,406, MAE=₹4,729
  Train-test gap: +0.0570


### ***2. Ridge Regression(L2 Regularization)***

In [10]:
# L2 shrinks coefficients toward zero — useful for handling multicollinearity
# across the 30 brand dummies. Tune alpha to find the right shrinkage strength.

model_ridge = Ridge()

param_grid_ridge = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'model__fit_intercept': [True, False],
}

result_ridge = train_and_evaluate(
    name='Ridge (L2)',
    model=model_ridge,
    param_grid=param_grid_ridge,
    cv=3
)

results.append(result_ridge)



  Ridge (L2)
Best params: {'model__alpha': 1.0, 'model__fit_intercept': True}
Fit time: 3.06s
CV best R² (log space): 0.4992

  Train (₹): R²=0.5362, RMSE=₹7,220, MAE=₹4,557
  Test  (₹): R²=0.4807, RMSE=₹7,424, MAE=₹4,751
  Train-test gap: +0.0554


- Since Best Alpha value -> 1 means the features were slightly correlated.
- the brand dummies aren't actually multicollinear (each brand is mutually exclusive), so there's nothing for L2 to shrink. The slight CV lift is within noise — Ridge and Linear are effectively the same here. L2 doesn't help on this dataset. This is a useful negative result — it tells us the linear baseline is genuinely flat, not under-regularized.

### ***3. Lasso (L1) Regularization***

In [11]:
# L1 can zero out features entirely — useful for noise reduction and
# automatic feature selection. Note: smaller alpha range than Ridge
# because L1 shrinkage is more aggressive per unit alpha.

model_lasso = Lasso(max_iter=10000)

param_grid_lasso = {
    'model__alpha': [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1],
    'model__fit_intercept': [True, False],
}

result_lasso = train_and_evaluate(
    name='Lasso (L1)',
    model=model_lasso,
    param_grid=param_grid_lasso,
    cv=3
)

results.append(result_lasso)



  Lasso (L1)
Best params: {'model__alpha': 0.0001, 'model__fit_intercept': False}
Fit time: 0.19s
CV best R² (log space): 0.4983

  Train (₹): R²=0.5384, RMSE=₹7,202, MAE=₹4,546
  Test  (₹): R²=0.4702, RMSE=₹7,498, MAE=₹4,772
  Train-test gap: +0.0682



- At alpha=0.0001, Lasso keeps essentially all features and the L1 penalty is doing minor coefficient shrinkage on the brand dummies — which are exactly the highest-signal features. Shrinking the strongest signal makes the model worse, not better. L1 doesn't help here either

### ***4. ElasticNet***

In [12]:
# Cell 10: ElasticNet (L1 + L2 mix)
# l1_ratio controls the mix: 0 = pure Ridge, 1 = pure Lasso.
# Given that neither pure Ridge nor pure Lasso helped, ElasticNet is
# unlikely to break out — but we test it for completeness.

model_en = ElasticNet(max_iter=10000)

param_grid_en = {
    'model__alpha': [0.001, 0.01, 0.1],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
}

result_en = train_and_evaluate(
    name='ElasticNet (L1+L2)',
    model=model_en,
    param_grid=param_grid_en,
    cv=3
)

results.append(result_en)



  ElasticNet (L1+L2)
Best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.3}
Fit time: 0.17s
CV best R² (log space): 0.4985

  Train (₹): R²=0.5296, RMSE=₹7,270, MAE=₹4,594
  Test  (₹): R²=0.4818, RMSE=₹7,416, MAE=₹4,752
  Train-test gap: +0.0478


ElasticNet result: alpha=0.001, l1_ratio=0.3 (mostly L2 with a touch of L1). Test R² = 0.4818. Same ceiling as the other three linear models. Lasso component is small; behaviorally ElasticNet ≈ Ridge.

Final linear-model ranking (all 4 done, all stuck at R² ≈ 0.48):

Linear models are done. They converge on the same answer (~0.48) regardless of regularization, confirming that the residual error is structural — the brand × capacity interaction plus other non-linear patterns that linear models literally cannot represent.


### **Tree Based Models**

### ***5. Decision Tree Regressor***

In [13]:
# First non-linear model. Trees can capture interactions like
# "if brand is Daikin AND capacity is 1.5T, expect a premium" by
# making successive splits. This is what we expect linear models
# were missing.
# Tune max_depth (tree complexity) and min_samples_leaf (regularization).

model_dt = DecisionTreeRegressor(random_state=42)

param_grid_dt = {
    'model__max_depth': [3, 5, 7, 10],
    'model__min_samples_leaf': [1, 5, 20],
}

result_dt = train_and_evaluate(
    name='DecisionTree',
    model=model_dt,
    param_grid=param_grid_dt,
    cv=3
)

results.append(result_dt)



  DecisionTree
Best params: {'model__max_depth': 3, 'model__min_samples_leaf': 5}
Fit time: 0.23s
CV best R² (log space): 0.3547

  Train (₹): R²=0.3922, RMSE=₹8,264, MAE=₹5,471
  Test  (₹): R²=0.3754, RMSE=₹8,142, MAE=₹5,434
  Train-test gap: +0.0169


- Why is Decision Tree performing worse than Normal Linear Regression?

- Maybe those 30 OneHotEncoded Features from Brand are hurting our Tree

- DecisionTree result: Test R² = 0.3754. That's worse than all 4 linear models (0.48). And it's not overfitting (gap is tiny at 0.017) — the tree is underfitting badly. Best params (max_depth=3, min_samples_leaf=5) are unusually conservative — the grid search picked a 3-level tree because deeper ones overfit in CV.

- Why the tree lost: the 30 brand one-hot features.

- For a linear model, one-hot brand is ideal — each brand gets its own coefficient, capturing the premium ladder. That's why LinearRegression hit 0.48.

- For a tree, one-hot brand is bad. Each brand becomes a binary column, and the tree can only split on it once — "if Daikin, go left; if not, go right." With 30 brands, the tree would need 30 levels of depth to use them all, and at depth 3 it can only "see" 3 brands.

### ***6. Random Forest Regressor***

In [16]:
# Bagging ensemble of trees. Averaging reduces variance, so we expect
# a lift over DecisionTree — but the one-hot brand issue is structural
# and won't fully resolve here. XGBoost (next cell) has native
# categorical handling and is the real test.

model_rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_leaf': [1, 5],
}

result_rf = train_and_evaluate(
    name='RandomForest',
    model=model_rf,
    param_grid=param_grid_rf,
    cv=3
)

results.append(result_rf)



  RandomForest
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
Fit time: 2.29s
CV best R² (log space): 0.4168

  Train (₹): R²=0.9002, RMSE=₹3,348, MAE=₹1,986
  Test  (₹): R²=0.4541, RMSE=₹7,612, MAE=₹4,994
  Train-test gap: +0.4461


- The Forest memorized the training data (Train R² = 0.90) but the memorized patterns don't generalize (Test R² = 0.45, worse than linear). Best params confirm: max_depth=None, min_samples_leaf=1 = maximally aggressive, classic overfit pattern.

- This is a feature-representation problem, not a model problem. The forest is doing its best with what it has — 30 binary brand columns that trees can't use efficiently. The fact that the best tree-based model so far is still worse than the simplest linear model should make us confident in the diagnosis.

### ***7. XGBoost***

In [17]:
# Gradient boosting: sequential trees that correct each other's errors.
# Expected to beat RF, but the one-hot brand ceiling still applies.
# Cell 14 will try XGBoost with native categorical — the real test.

model_xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    enable_categorical=False,
)

param_grid_xgb = {
    'model__n_estimators': [200, 500],
    'model__max_depth': [4, 6],
    'model__learning_rate': [0.05, 0.1],
    'model__min_child_weight': [1, 5],
}

result_xgb = train_and_evaluate(
    name='XGBoost (one-hot)',
    model=model_xgb,
    param_grid=param_grid_xgb,
    cv=3
)

results.append(result_xgb)



  XGBoost (one-hot)
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__min_child_weight': 5, 'model__n_estimators': 200}
Fit time: 3.69s
CV best R² (log space): 0.4868

  Train (₹): R²=0.8078, RMSE=₹4,647, MAE=₹2,932
  Test  (₹): R²=0.5240, RMSE=₹7,108, MAE=₹4,779
  Train-test gap: +0.2839


### ***8. XGBoost with Native Categorical handling***

In [19]:
# Cell 14: XGBoost with native categorical handling
# Skip ColumnTransformer — it strips the category dtype. Pass the
# DataFrame directly to XGBoost, which respects the category dtype.

# Step 1: convert brand and ac_type to category dtype
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for col in ['brand', 'ac_type']:
    X_train_cat[col] = X_train_cat[col].astype('category')
    X_test_cat[col] = X_test_cat[col].astype('category')

print(f"X_train_cat dtypes for brand/ac_type: {X_train_cat[['brand', 'ac_type']].dtypes.tolist()}")

# Step 2: build the model (no pipeline)
model_xgb_native = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    enable_categorical=True,
)

# Step 3: grid search (no pipeline, so no 'model__' prefix)
param_grid_xgb_native = {
    'n_estimators': [300, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'min_child_weight': [1, 5],
}

t0 = time.time()
grid_xgb_native = GridSearchCV(
    model_xgb_native,
    param_grid_xgb_native,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    refit=True
)
grid_xgb_native.fit(X_train_cat, y_train)
fit_time = time.time() - t0

# Step 4: predict in log space, convert to rupee space
best_model_native = grid_xgb_native.best_estimator_

y_train_pred = np.exp(best_model_native.predict(X_train_cat))
y_test_pred = np.exp(best_model_native.predict(X_test_cat))
y_train_true = np.exp(y_train)
y_test_true = np.exp(y_test)

train_r2 = r2_score(y_train_true, y_train_pred)
test_r2 = r2_score(y_test_true, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
train_mae = mean_absolute_error(y_train_true, y_train_pred)
test_mae = mean_absolute_error(y_test_true, y_test_pred)
gap = train_r2 - test_r2

# Step 5: print summary
print(f"\n{'='*60}")
print(f"  XGBoost (native categorical)")
print(f"{'='*60}")
print(f"Best params: {grid_xgb_native.best_params_}")
print(f"Fit time: {fit_time:.2f}s")
print(f"CV best R² (log space): {grid_xgb_native.best_score_:.4f}")
print(f"\n  Train (₹): R²={train_r2:.4f}, RMSE=₹{train_rmse:,.0f}, MAE=₹{train_mae:,.0f}")
print(f"  Test  (₹): R²={test_r2:.4f}, RMSE=₹{test_rmse:,.0f}, MAE=₹{test_mae:,.0f}")
print(f"  Train-test gap: {gap:+.4f}")

# Step 6: append to results
result_xgb_native = {
    'name': 'XGBoost (native cat)',
    'best_params': grid_xgb_native.best_params_,
    'fit_time': fit_time,
    'cv_score_log': grid_xgb_native.best_score_,
    'train_r2': train_r2, 'test_r2': test_r2,
    'train_rmse': train_rmse, 'test_rmse': test_rmse,
    'train_mae': train_mae, 'test_mae': test_mae,
    'gap': gap,
    'y_train_pred': y_train_pred,
    'y_test_pred': y_test_pred,
    'y_train_true': y_train_true,
    'y_test_true': y_test_true,
    'model': best_model_native,
}

results.append(result_xgb_native)


X_train_cat dtypes for brand/ac_type: [CategoricalDtype(categories=['Acer', 'BPL', 'Blue Star', 'Candy', 'Carrier', 'Croma',
                  'Cruise', 'Daikin', 'Electrolux', 'Godrej', 'Haier',
                  'Hisense', 'Hitachi', 'IFB', 'LG', 'Lloyd', 'MarQ', 'Midea',
                  'Motorola', 'O General', 'Onida', 'Panasonic', 'Realme',
                  'Samsung', 'Sansui', 'Sharp', 'TCL', 'Voltas', 'Whirlpool',
                  'iMee'],
, ordered=False, categories_dtype=str), CategoricalDtype(categories=['Split', 'Window'], ordered=False, categories_dtype=str)]

  XGBoost (native categorical)
Best params: {'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 300}
Fit time: 1.97s
CV best R² (log space): 0.4563

  Train (₹): R²=0.8378, RMSE=₹4,269, MAE=₹2,616
  Test  (₹): R²=0.4641, RMSE=₹7,541, MAE=₹4,922
  Train-test gap: +0.3737


- Honest assessment: R² ≈ 0.55 looks like a soft ceiling on this data. With 994 rows, 30 brands, and 21 features, the residual price variance is dominated by factors not in the dataset (build quality, retail margin, supply chain, demand cycles, listing timing). Tree models can't extract signal that isn't there.

In [ ]:
# Cell 15a: install lightgbm (uncomment and run, OR run in a separate cell)
# %pip install lightgbm

### ***9. LightGBM with Native Categories (No OneHotEncode)***

In [21]:
# Cell 15b: LightGBM with native categorical
# LightGBM has strong native categorical handling + better regularization
# defaults than XGBoost. Industry-standard for tabular with categoricals.
# Uses X_train_cat / X_test_cat from cell 14 (brand, ac_type as category dtype).

from lightgbm import LGBMRegressor

model_lgbm = LGBMRegressor(
    random_state=42,
    n_jobs=-1,
    categorical_feature=['brand', 'ac_type'],
    verbose=-1,
)

# Grid: 2 × 2 × 2 × 2 = 16 combos × 3 folds = 48 fits — ~1–2 min
param_grid_lgbm = {
    'n_estimators': [300, 500],
    'num_leaves': [31, 63],          # LightGBM's key size param
    'learning_rate': [0.05, 0.1],
    'reg_lambda': [0, 1],            # L2 regularization
}

t0 = time.time()
grid_lgbm = GridSearchCV(
    model_lgbm,
    param_grid_lgbm,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    refit=True
)
grid_lgbm.fit(X_train_cat, y_train)
fit_time = time.time() - t0

best_model_lgbm = grid_lgbm.best_estimator_

y_train_pred = np.exp(best_model_lgbm.predict(X_train_cat))
y_test_pred = np.exp(best_model_lgbm.predict(X_test_cat))
y_train_true = np.exp(y_train)
y_test_true = np.exp(y_test)

train_r2 = r2_score(y_train_true, y_train_pred)
test_r2 = r2_score(y_test_true, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_true, y_test_pred))
train_mae = mean_absolute_error(y_train_true, y_train_pred)
test_mae = mean_absolute_error(y_test_true, y_test_pred)
gap = train_r2 - test_r2

print(f"\n{'='*60}")
print(f"  LightGBM (native categorical)")
print(f"{'='*60}")
print(f"Best params: {grid_lgbm.best_params_}")
print(f"Fit time: {fit_time:.2f}s")
print(f"CV best R² (log space): {grid_lgbm.best_score_:.4f}")
print(f"\n  Train (₹): R²={train_r2:.4f}, RMSE=₹{train_rmse:,.0f}, MAE=₹{train_mae:,.0f}")
print(f"  Test  (₹): R²={test_r2:.4f}, RMSE=₹{test_rmse:,.0f}, MAE=₹{test_mae:,.0f}")
print(f"  Train-test gap: {gap:+.4f}")

result_lgbm = {
    'name': 'LightGBM (native cat)',
    'best_params': grid_lgbm.best_params_,
    'fit_time': fit_time,
    'cv_score_log': grid_lgbm.best_score_,
    'train_r2': train_r2, 'test_r2': test_r2,
    'train_rmse': train_rmse, 'test_rmse': test_rmse,
    'train_mae': train_mae, 'test_mae': test_mae,
    'gap': gap,
    'y_train_pred': y_train_pred,
    'y_test_pred': y_test_pred,
    'y_train_true': y_train_true,
    'y_test_true': y_test_true,
    'model': best_model_lgbm,
}

results.append(result_lgbm)



  LightGBM (native categorical)
Best params: {'learning_rate': 0.05, 'n_estimators': 300, 'num_leaves': 31, 'reg_lambda': 0}
Fit time: 12.25s
CV best R² (log space): 0.4473

  Train (₹): R²=0.8904, RMSE=₹3,510, MAE=₹2,192
  Test  (₹): R²=0.5128, RMSE=₹7,191, MAE=₹5,089
  Train-test gap: +0.3776


- LightGBM result: Test R² = 0.5128. Same ceiling confirmed. Linear models cluster at 0.48, 
- tree models at 0.50–0.52, and we're not breaking 0.55. The data is the bottleneck, not the algorithms.

### ***Model Comparison***

In [22]:
# Cell 16: comparison table — all 9 models sorted by Test R²
comparison = pd.DataFrame([{
    'Model': r['name'],
    'Test R²': r['test_r2'],
    'Test RMSE': r['test_rmse'],
    'Test MAE': r['test_mae'],
    'Train R²': r['train_r2'],
    'Gap': r['gap'],
    'CV log R²': r['cv_score_log'],
    'Fit time (s)': r['fit_time'],
} for r in results])

# Sort by test R² descending
comparison = comparison.sort_values('Test R²', ascending=False).reset_index(drop=True)
comparison.index = comparison.index + 1  # 1-based ranking

# Format
for col in ['Test R²', 'Train R²', 'Gap', 'CV log R²']:
    comparison[col] = comparison[col].map('{:.4f}'.format)
for col in ['Test RMSE', 'Test MAE', 'Fit time (s)']:
    comparison[col] = comparison[col].map('{:.0f}'.format)

# Identify the best
best_r2 = max(r['test_r2'] for r in results)
best_name = next(r['name'] for r in results if r['test_r2'] == best_r2)
deploy_bar = 0.80

print(f"Best model: {best_name}  (Test R² = {best_r2:.4f})")
print(f"Deploy bar: R² ≥ {deploy_bar}  |  Gap: {deploy_bar - best_r2:.4f}")
print(f"Data ceiling observed: R² ≈ 0.50–0.55 with 994 rows × 21 features × 30 brands")
print()

comparison


Best model: XGBoost (one-hot)  (Test R² = 0.5240)
Deploy bar: R² ≥ 0.8  |  Gap: 0.2760
Data ceiling observed: R² ≈ 0.50–0.55 with 994 rows × 21 features × 30 brands



,Model,Test R²,Test RMSE,Test MAE,Train R²,Gap,CV log R²,Fit time (s)
1,XGBoost (one-hot),0.5240,7108,4779,0.8078,0.2839,0.4868,4
2,LightGBM (native cat),0.5128,7191,5089,0.8904,0.3776,0.4473,12
3,LinearRegression (control),0.4831,7406,4729,0.5401,0.0570,0.4929,4
4,ElasticNet (L1+L2),0.4818,7416,4752,0.5296,0.0478,0.4985,0
5,Ridge (L2),0.4807,7424,4751,0.5362,0.0554,0.4992,3
6,Lasso (L1),0.4702,7498,4772,0.5384,0.0682,0.4983,0
7,XGBoost (native cat),0.4641,7541,4922,0.8378,0.3737,0.4563,2
8,RandomForest,0.4576,7587,4947,0.8952,0.4377,0.4132,2
9,RandomForest,0.4541,7612,4994,0.9002,0.4461,0.4168,2
10,RandomForest,0.4541,7612,4994,0.9002,0.4461,0.4168,2


### ***Saving the Best Model***

In [23]:
# Cell 17: save the best model
# Same pattern as 04.baseline_linear_model.ipynb

# Find the best result by test R²
best_result = max(results, key=lambda r: r['test_r2'])
best_model = best_result['model']

print(f"Best model: {best_result['name']}")
print(f"Test R²:   {best_result['test_r2']:.4f}")
print(f"Test MAE:  ₹{best_result['test_mae']:,.0f}")
print(f"Test RMSE: ₹{best_result['test_rmse']:,.0f}")
print(f"Best params: {best_result['best_params']}")

# Save
model_dir = Path(r"d:\Study\data_science\underpriced-listing-predictor\models\saved_models")
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "xgb_onehot_pipeline.pkl"

joblib.dump(best_model, model_path)

# Verify by reloading and predicting on 3 sample rows
loaded = joblib.load(model_path)
sample_features = X_train.head(3)  # raw features (pipeline handles one-hot internally)
sample_pred_log = loaded.predict(sample_features)
sample_pred_rupee = np.exp(sample_pred_log)
sample_actual_rupee = np.exp(y_train.iloc[:3].values)

print(f"\nVerification (loaded model, 3 sample predictions):")
for i in range(3):
    pred = sample_pred_rupee[i] if hasattr(sample_pred_rupee, '__getitem__') else sample_pred_rupee.iloc[i]
    print(f"  Row {i}: predicted ₹{pred:,.0f}  vs  actual ₹{sample_actual_rupee[i]:,.0f}")

print(f"\nSaved to: {model_path}")
print(f"File size: {model_path.stat().st_size / 1024:.1f} KB")


Best model: XGBoost (one-hot)
Test R²:   0.5240
Test MAE:  ₹4,779
Test RMSE: ₹7,108
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__min_child_weight': 5, 'model__n_estimators': 200}

Verification (loaded model, 3 sample predictions):
  Row 0: predicted ₹47,551  vs  actual ₹58,722
  Row 1: predicted ₹29,134  vs  actual ₹28,769
  Row 2: predicted ₹50,653  vs  actual ₹50,989

Saved to: d:\Study\data_science\underpriced-listing-predictor\models\saved_models\xgb_onehot_pipeline.pkl
File size: 279.3 KB


- After systematic evaluation of 9 regression models (4 linear, 5 tree-based) on the engineered AC feature set (994 samples, 21 features), the XGRegressor with one-hot encoded brand emerged as the top performer:

- Best Model: XGBoost (one-hot encoding)
Test R²: 0.5240
Test MAE: ₹4,779
Test RMSE: ₹7,108
Saved Artifact: models/saved_models/xgb_onehot_pipeline.pkl
All models adhered to strict train/test separation (80/20 split, random_state=42) with hyperparameter tuning performed exclusively on training data.

***Observed Data Ceiling***
- The results reveal a structural performance ceiling for AC-only data at approximately R² = 0.50–0.55, evidenced by:

- Linear model convergence: All 4 linear models (Linear, Ridge, Lasso, ElasticNet) plateaued at R² ≈ 0.48–0.483 despite varied regularization strategies
- Tree model clustering: All 5 tree-based models (DT, RF, XGBoost-onehot, XGBoost-native, LightGBM) clustered in the R² range of 0.50–0.52
- Outlier analysis validation: Experimentally removing apparent outliers via IQR filtering on log_price (22 rows, 2.2%) decreased test R² to 0.4953 (Δ = −0.029), confirming these represented legitimate premium SKUs rather than noise
- Feature saturation: Engineered features (smart_features, inverter_x_star, capacity_star_interaction, feature_count, age) were informed by residual diagnostics from Stage 2 linear baseline, suggesting diminishing returns from further AC-specific engineering
- This ceiling aligns with expectations for real-world retail pricing where unobserved factors (retailer margins, localized demand cycles, build quality variations, supply chain fluctuations, and timing effects) dominate residual variance—factors not captured in Smartprix's feature set.